# Iterators and generators

Iteration retrieves values from an object one at a time. Iterables provide iterators; iterators track traversal state; generators implement the iterator protocol through suspended execution.

The examples use Python 3 without external packages or datasets.

## Iterables and iterators

An **iterable** is an object from which `iter()` obtains an iterator. Lists, tuples, sets, and dictionaries are iterables. An **iterator** tracks progress and supplies one value at a time through `next()`. A `for` loop obtains an iterator and consumes it until `StopIteration` signals exhaustion.

Two iterators over the same list have independent positions. Calling `iter()` on an iterator returns that same iterator without resetting its position.

In [1]:
topics = ["basics", "pandas", "databases"]
first = iter(topics)
second = iter(topics)

print(next(first))   # basics
print(next(first))   # pandas
print(next(second))  # basics: independent position
print(iter(first) is first)  # True
print(list(first))   # ['databases']: only the remaining items
print(list(first))   # []: now exhausted
print(list(topics))  # the original list still contains all three items

basics
pandas
basics
True
['databases']
[]
['basics', 'pandas', 'databases']


## Exhaustion and default values

An exhausted iterator raises `StopIteration`. The two-argument form `next(iterator, default)` returns the default value when no items remain.

A fresh iterator over a list starts a new traversal. An exhausted iterator remains exhausted.

In [2]:
empty = iter([])
try:
    next(empty)
except StopIteration:
    print("No items left")

print(next(empty, "finished"))
print(list(iter(topics)))

No items left
finished
['basics', 'pandas', 'databases']


## The iterator protocol

An iterator implements two special methods:

- `__iter__()` returns the iterator itself. Calling `iter(iterator)` therefore preserves its identity and current position.
- `__next__()` returns the next value or raises `StopIteration` when exhausted. Calling `next(iterator)` invokes this behavior.

Once exhausted, an iterator must continue to raise `StopIteration` on later requests. `None` is a possible data value, not an exhaustion signal.

An iterable can instead return a fresh iterator from its `__iter__()` method, as lists do. This separates a reusable collection from an individual traversal. Python also supports a legacy sequence fallback for some objects with integer-indexed `__getitem__()`.

Generators implement the iterator protocol automatically; a custom class makes the same state transitions explicit. In the example, `self` refers to the current iterator object, and `__init__` initializes its state.

In [3]:
class CountdownIterator:
    """Iterator yielding integers from start down to one."""

    def __init__(self, start):
        self.remaining = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.remaining <= 0:
            raise StopIteration
        value = self.remaining
        self.remaining -= 1
        return value


counter = CountdownIterator(3)
print(iter(counter) is counter)  # True
print(next(counter))             # 3
print(list(counter))             # [2, 1]
print(next(counter, "finished")) # finished
print(next(counter, "finished")) # finished: exhaustion persists
print(list(CountdownIterator(0))) # []

True
3
[2, 1]
finished
finished
[]


## Exhaustion is separate from data values

An iterator can yield `None` and still contain more values. A `for` loop ends in response to `StopIteration`, not in response to a particular returned value.

`next(iterator, default)` replaces the exhaustion exception with a default result. A default that can also occur in the data is ambiguous; a unique sentinel object distinguishes the two cases.

In [4]:
values = iter([None, "last"])
sentinel = object()
first_value = next(values, sentinel)
print(first_value is None)       # True: a real item
print(first_value is sentinel)   # False
print(next(values, sentinel))    # last
print(next(values, sentinel) is sentinel)  # True: exhausted

True
False
last
True


## A generator function uses `yield`

A function containing `yield` creates a generator when called. A generator is an iterator. Its body starts when a value is requested; `yield` supplies that value and suspends execution while preserving local state. The next request resumes after the previous `yield`.

Reaching the end of the function, or executing `return`, ends iteration. A generator can contain both `yield` and `return`.

In [5]:
def countdown(start):
    """Yield integers from start down to 1; yield nothing for start <= 0."""
    while start > 0:
        print(f"Producing {start}")
        yield start
        start -= 1


remaining = countdown(3)
print("Generator created; its body has not run yet")
print("Received:", next(remaining))
print("Rest:", list(remaining))
print("Again:", list(remaining))  # []
print("Fresh generator:", list(countdown(2)))

Generator created; its body has not run yet
Producing 3
Received: 3
Producing 2
Producing 1
Rest: [2, 1]
Again: []
Producing 2
Producing 1
Fresh generator: [2, 1]


## List comprehensions and generator expressions

A list comprehension builds its result immediately. A generator expression computes values as the consumer requests them. The expressions below produce the same sequence of values, but the generator is consumed once.

Lazy evaluation avoids storing all generated results at once. An iterator over an existing list still retains access to that list, and `list(generator)` materializes all remaining results. Lists support indexing and repeated traversal; generators support incremental consumption.

In [6]:
squares_list = [number ** 2 for number in range(5)]
squares_generator = (number ** 2 for number in range(5))

print(squares_list)
print(list(squares_generator))
print(list(squares_generator))  # []

# sum consumes values incrementally without building a list of squares.
print(sum(number ** 2 for number in range(5)))  # 30

[0, 1, 4, 9, 16]
[0, 1, 4, 9, 16]
[]
30


## Lazy data-processing pipelines

Generators can form successive processing stages. In this example, `parse_measurements` skips blank strings and converts the remaining strings to floats. A generator expression filters nonnegative values, and `sum()` consumes the pipeline.

The sample input is already a list in memory. The pipeline avoids additional lists for cleaned and filtered results. Invalid nonblank numbers raise `ValueError`.

In [7]:
def parse_measurements(lines):
    """Yield a float for each nonblank input line."""
    for line in lines:
        stripped = line.strip()
        if stripped:
            yield float(stripped)


raw_measurements = [" 12.5 ", "", "-2.0", " 7.5", "   ", "0"]
measurements = parse_measurements(raw_measurements)
nonnegative = (value for value in measurements if value >= 0)

print(sum(nonnegative))       # 20.0
print(list(nonnegative))      # []: sum consumed the pipeline
print(list(parse_measurements(raw_measurements)))  # recreate it to reuse it

20.0
[]
[12.5, -2.0, 7.5, 0.0]


## State retained between yields

A running-total generator retains a local accumulator between requests. Each input value updates the accumulator and produces one cumulative total.

The implementation accepts any iterable of compatible numeric values, including another generator. Empty input produces no values. Only the current total is retained by the accumulator; no result list is built inside the function.

In [8]:
def running_totals(values):
    """Yield cumulative totals without materializing the input."""
    total = 0
    for value in values:
        total += value
        yield total


print(list(running_totals([3, 1, 4])))  # [3, 4, 8]
print(list(running_totals([])))        # []
print(list(running_totals(iter([2, -3, 5]))))  # [2, -1, 4]
print(list(running_totals(parse_measurements(raw_measurements))))

[3, 4, 8]
[]
[2, -1, 4]
[12.5, 10.5, 18.0, 18.0]


## Consumption and reuse

A generator's state advances whenever a consumer requests an item. Consumers include `next()`, `for`, `list()`, and `sum()`. Partial consumption leaves only the remaining values available; full consumption exhausts the generator.

Calling a generator function again creates a new generator with independent execution state. Reusing an exhausted generator does not repeat the computation.